In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

**Load your student performance data:**

In [ ]:
import pandas as pd
file_path_1 = "../data/student_academic_data_anonymized.csv"
file_path_2 = "../data/student_survey_data_anonymized.csv"
data1 = pd.read_csv(file_path_1)
data2 = pd.read_csv(file_path_2)

**MERGE TWO CSV FILES**

In [ ]:
c2 = data2.columns
for i in c2:
  data1[i] = data2[i]
data = data1

In [ ]:
# Rename relevant columns for clarity
data.rename(columns={
    'Physics': 'Physics_Score',
    'Chemistry': 'Chemistry_Score',
    'Maths': 'Maths_Score'
}, inplace=True)


**Preprocess the data:**

In [ ]:
# Compute total number of arrears
arrears_columns = ['NO OF ARREARS S1', 'NO OF ARREARS S2', 'NO OF ARREARS S3', 'NO OF ARREARS S4', 'NO OF ARREARS S5']
data['Total_Arrears'] = data[arrears_columns].sum(axis=1)

# Compute total credits earned
credits_columns = ['CREDITS EARNED S1', 'CREDITS EARNED S2', 'CREDITS EARNED S3', 'CREDITS EARNED S4', 'CREDITS EARNED S5']
data['Total_Credits_Earned'] = data[credits_columns].sum(axis=1)

In [ ]:
# List of columns to check
from sklearn.preprocessing import LabelEncoder
col_name = list(data.columns)  # Specify the categorical columns
col_name.remove("Name")
col_name.remove("Student ID")
col_name.remove("University No")
# Apply LabelEncoder to string columns
label_encoders = {}  # Dictionary to store encoders for later use

for col in col_name:
    if data[col].dtype == 'object':  # Check if column has string values
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])  # Transform the column
        label_encoders[col] = le  # Save encoder if you need to decode later

In [ ]:
print(col_name)

In [ ]:
data

In [ ]:
from sklearn.preprocessing import StandardScaler

# Selecting relevant numerical columns for clustering
attendance_columns = [col for col in data.columns if col.endswith('_Atten %')]
assignment_columns = [col for col in data.columns if col.endswith('_Assign')]
university_results_columns = [col for col in data.columns if col.startswith('UR_')]

# Social relevance columns (ALL QUESTIONS INCLUDED)
social_relevance_columns = [
    "1. What is the highest level of education completed by your parents?",
    "3. Do your parents have any influence on your academic decisions?",
    "4. How often do you discuss your academic progress with your parents?",
    "5. Where do you reside?",
    "6. What is your current relationship status?",
    "7. Do you receive any scholarship from your college?",
    "8. Do you receive any financial assistance from the government?",
    "9. How do you manage your finances while attending college?",
    "10. Have you ever had to take a loan to finance your education?",
    "11. How often do you feel stressed or overwhelmed due to college-related pressures?",
    "12. Have you ever sought help from a counselor or mental health professional due to college-related stress?",
    "13. How would you rate your overall mental well-being?",
    "14. How important is social media in your life?",
    "15. Have you ever experienced or witnessed bullying or harassment on campus?",
    "16. Do you think your college does enough to promote diversity, equity, and inclusion?"
]

important_columns = ['Physics_Score', 'Chemistry_Score', 'Maths_Score', 'Total_Arrears', 'Total_Credits_Earned']

# Check if social relevance columns are present in the DataFrame
social_relevance_columns_present = [col for col in social_relevance_columns if col in data.columns]

# Include social relevance columns only if they are present
selected_columns = important_columns + attendance_columns + assignment_columns + university_results_columns + social_relevance_columns_present
df_cluster = data[selected_columns].copy()

# Handling missing values by filling with mean
df_cluster.fillna(df_cluster.mean(), inplace=True)

# Standardizing the data
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

3. **K-Means Clustering**

**Apply K-Means clustering:**

In [ ]:
from sklearn.cluster import KMeans
# Applying K-Means clustering with 5 clusters (logical basis)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data['cluster'] = kmeans.fit_predict(df_scaled) + 1

**Add cluster labels to your data:**

In [ ]:
# Define cluster labels
cluster_labels = {
    1: "High Achievers",
    2: "Moderate Performers",
    3: "Slow Learners",
    4: "Inconsistent Performers",
    5: "Potential Improvers"
}

# Assign labels to clusters
data['cluster_label'] = data['cluster'].map(cluster_labels)

In [ ]:
'''cluster_order = ["High Achievers", "Moderate Performers", "Slow Learners", "Inconsistent Performers", "Potential Improvers"]
cluster_mapping = {i+1: label for i, label in enumerate(cluster_order)}
data['cluster_label'] = data['cluster'].map(cluster_labels)'''

4**. Recommendations**

Analyze cluster characteristics:

In [ ]:
for i in range(1,6):
  print(len(data[data['cluster'] == i]))

In [ ]:
data[data['cluster'] == 5]

In [ ]:
for i in range(1,6):
        cluster_data = data[data['cluster'] == i]
        print(f"Cluster {i}:")
        print(cluster_data.describe())  # Explore the characteristics of each cluster

**Provide recommendations based on cluster:**

In [ ]:
recommendations = {
    1: "- Participate in advanced coursework and research projects.\n- Take up leadership roles in student communities.\n- Seek mentorship and industry exposure to expand career opportunities.\n- Focus on networking and internships for practical experience.",
    2: "- Attend skill development workshops and career counseling sessions.\n- Improve time management and study planning.\n- Engage in peer learning and collaborative projects.\n- Apply for scholarships and financial aid if needed.",
    3: "- Enroll in remedial classes and seek academic counseling.\n- Take advantage of financial assistance programs.\n- Participate in mental health programs and stress management workshops.\n- Seek tutoring and one-on-one mentorship for academic improvement.",
    4: "- Identify the reasons for inconsistent performance and work on study habits.\n- Attend soft skills training and career orientation programs.\n- Seek guidance from mental health professionals if needed.\n- Develop personalized study schedules and set achievable goals.",
    5: "- Focus on structured academic improvement plans.\n- Participate in motivational workshops and mentorship programs.\n- Seek financial guidance and explore scholarship opportunities.\n- Engage in extracurricular activities to boost confidence and teamwork skills."
}

# Map recommendations to clusters
data['recommendations'] = data['cluster'].map(recommendations)

**Prepare data for Random Forest classification**


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)
X = df_scaled
y = data['cluster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Print classification report
print(classification_report(y_test, y_pred))


In [ ]:
# Print students' names in each cluster
#for cluster in range(5):
   # print(f"\nStudents in Cluster {cluster} ({cluster_labels[cluster]}):")
    #cluster_students = data[data['cluster'] == cluster]['Name'].tolist()
    #for student in cluster_students:
       # print(student)

# Display a sample of the results
#print(data[['cluster', 'cluster_label', 'recommendation']].head(10))


In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

**CLUSTER COUNT**


In [ ]:
# Get the count of unique students in each cluster
cluster_counts = data.groupby('cluster')['Name'].nunique()
# Print the cluster counts
print("Total Students in Each Cluster:")
print(cluster_counts)
 # Calculate the total number of unique students across all clusters
total_students = cluster_counts.sum()

# Print the total number of students
print("\nTotal Students Across All Clusters:", total_students)

**USER INPUT**

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create input widget for student name
student_name_input = widgets.Text(
    placeholder='Enter student name',
    description='Student Name:',
    disabled=False
)

# Create output widget to display results
output_widget = widgets.Output()

# Define function to handle button click
def on_button_clicked(b):
    with output_widget:
        output_widget.clear_output()  # Clear previous output
        student_name = student_name_input.value
        student_info = get_student_info(student_name)

        if student_info:
            cluster, cluster_name, recommendation = student_info
            print(f"Cluster: {cluster} ({cluster_name})")
            print(f"Recommendation: {recommendation}")
        else:
            print("Student not found.")

# Create button widget
button = widgets.Button(description="Get Info")
button.on_click(on_button_clicked)

# Display widgets
display(student_name_input, button, output_widget)

**FRONT END**

In [ ]:
%pip install gradio


**Simple Web UI for ML Model**

In [ ]:
print(data.columns)
data.columns = data.columns.str.strip()



In [ ]:
student_column = [col for col in data.columns if "student" in col.lower()]
print("Detected Student Column:", student_column)


In [ ]:
def plot_performance_graph(name):
    student = data[data['Name'] == name]
    if student.empty:
        return "Student not found"

    s1_to_s5 = student[arrears_columns].values.flatten()
    plt.figure(figsize=(6, 5))
    plt.barplot(["S1", "S2", "S3", "S4", "S5"], s1_to_s5, marker='o', linestyle='-')
    plt.title(f"Performance Trend for {name}")
    plt.xlabel("Semester")
    plt.ylabel("Arrears")
    plt.grid()
    plt.show()

In [ ]:
def show_student_performance(name):
    student = data[data['Name'] == name]
    if student.empty:
        return "Student not found"


    Semesters = ['S1', 'S2', 'S3', 'S4', 'S5']
    Sgpa = [student_row[col].values[0] for col in Semesters]

    plt.figure(figsize=(6, 4))
    sns.plot(x=Semesters, y=Sgpa, palette='viridis')
    plt.title(f"Semester Performance for {name}")
    plt.ylabel("SGPA")
    plt.show()

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load dataset
import pandas as pd
#file_path_1 = "../data/student_academic_data_anonymized.csv"
#file_path_2 = "../data/student_survey_data_anonymized.csv"
#data1 = pd.read_csv(file_path_1)
#data2 = pd.read_csv(file_path_2)

# Merge datasets on 'Name'
#data = pd.merge(data1, data2, on='Name', how='outer')

# Rename columns to match function expectations
data.rename(columns={
    "NO OF ARREARS S1": "S1",
    "NO OF ARREARS S2": "S2",
    "NO OF ARREARS S3": "S3",
    "NO OF ARREARS S4": "S4",
    "NO OF ARREARS S5": "S5",
    "SGPA1": "Sem 1",
    'SGPA2': 'Sem 2',
    'SGPA3': 'Sem 3',
    'SGPA4': 'Sem 4',
    'SGPA 5': 'Sem 5'
}, inplace=True)

# Convert academic performance columns to numeric
required_columns = ['Name', 'S1', 'S2', 'S3', 'S4', 'S5', 'Sem 1', 'Sem 2', 'Sem 3', 'Sem 4', 'Sem 5']
#for col in required_columns[1:]:
    #data[col] = pd.to_numeric(data[col], errors='coerce')

# Fill missing values with column mean
numerical_cols = data.select_dtypes(include=np.number).columns
data[numerical_cols] = data[numerical_cols].fillna(data[numerical_cols].mean())

# Standardize numerical features
scaler = StandardScaler()
df_scaled = scaler.fit_transform(data[required_columns[1:]])

# Apply K-Means clustering
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data['cluster'] = kmeans.fit_predict(df_scaled) + 1

# Define cluster labels
cluster_labels = {
    1: "High Achievers",
    2: "Moderate Performers",
    3: "Slow Learners",
    4: "Inconsistent Performers",
    5: "Potential Improvers"
}
data['cluster_label'] = data['cluster'].map(cluster_labels)

# Define recommendations in bullet points
recommendations = {
    1: "📌 **High Achievers (Top Performers)**\n"
       "✅ Engage in research projects, Olympiads, and hackathons.\n"
       "✅ Take up mentoring roles for peers or juniors.\n"
       "✅ Contribute to community service initiatives, such as teaching underprivileged children.\n"
       "✅ Start networking with industry professionals and apply for internships in top firms.",

    2: "📌 **Moderate Performers (Good, but Needs Pushing)**\n"
       "✅ Improve through time management & focused study plans.\n"
       "✅ Join student organizations, participate in group discussions.\n"
       "✅ Engage in eco-clubs, volunteer work, or awareness campaigns.\n"
       "✅ Take skill-enhancing online courses & certifications to improve employability.",

    3: "📌 **Slow Learners (Needs Personalized Support)**\n"
       "✅ Seek extra tutoring, one-on-one mentorship, and structured study plans.\n"
       "✅ Schools should provide counseling and emotional support.\n"
       "✅ Encourage participation in team-based activities to build confidence.\n"
       "✅ Provide vocational training, soft skills workshops, and internship opportunities.",

    4: "📌 **Inconsistent Performers (Erratic Performance, Needs Stability)**\n"
       "✅ Identify gaps in knowledge and work on consistent study habits.\n"
       "✅ Encourage taking responsibility in group projects to develop accountability.\n"
       "✅ Involve in debates, Model UNs, or community projects to gain exposure.\n"
       "✅ Improve by taking mock interviews and attending career counseling sessions.",

    5: "📌 **Potential Improvers (Students with Growth Potential)**\n"
       "✅ Work on study techniques, peer learning, and smart revision.\n"
       "✅ Take initiative in small leadership roles like class monitors or event coordinators.\n"
       "✅ Participate in community-driven innovation challenges or skill-sharing programs.\n"
       "✅ Build networking skills, attend industry webinars, and develop a career plan."
}


# Function to get student info and generate plots
def get_student_info(student_name):
    student_row = data[data['Name'].str.strip().str.lower() == student_name.strip().lower()]

    if student_row.empty:
        return "Student not found.", "", "", None, None

    cluster = int(student_row['cluster'].values[0])
    cluster_name = student_row['cluster_label'].values[0]
    recommendation = recommendations[cluster]
###
    from matplotlib.ticker import MaxNLocator

    semesters = ['S1', 'S2', 'S3', 'S4', 'S5']
    scores = [student_row[col].values[0] for col in semesters]

    fig1, ax1 = plt.subplots(figsize=(6, 6))  # Reduced si
    ax1.bar(semesters, scores, color='b')  # Bar plot
    ax1.set_xlabel('Semester')
    ax1.set_ylabel('Arrears')
    ax1.set_title(f'No of Arrears: {student_name}')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)  # Grid only on y-axis
    ax1.set_ylim(0, 10)
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))  # Ensure integer values on y-axis
    plt.xticks(rotation=45, ha="right", rotation_mode="anchor")
    performance_plot = fig1
###

    ####
    Semesters = ['Sem 1', 'Sem 2', 'Sem 3', 'Sem 4', 'Sem 5']
    Sgpa = [student_row[col].values[0] for col in Semesters]

    fig2, ax2 = plt.subplots(figsize=(6, 6))  # ✅ Reduced size
    ax2.plot(Semesters, Sgpa, marker='o', linestyle='-', color='b')  # Line plot
    ax2.set_xlabel('Semester')
    ax2.set_ylabel('SGPA')
    ax2.set_ylim(0, 10)
    ax2.set_title(f'Semester Performance: {student_name}')
    Sgpa = fig2
    #####

    return cluster, cluster_name, recommendation, performance_plot, Sgpa


# Custom CSS for Scrolling Fix & Layout
custom_css = """
/* 🌟 Soft Violet Gradient Background */
html, body {
    min-height: 100vh;
    overflow-y: auto;
    overflow-x: hidden;
    background: linear-gradient(-45deg, #2D1B4C, #3B2466, #4A2F80, #2D1B4C);
    background-size: 300% 300%;
    animation: gradientFlow 12s infinite alternate ease-in-out;
}

/* 🎭 Smooth Background Animation */
@keyframes gradientFlow {
    0% { background-position: 0% 50%; }
    100% { background-position: 100% 50%; }
}

/* 📦 Elegant Container */
.gradio-container {
    width: 100%;
    min-height: 100vh;
    padding: 25px;
    border-radius: 12px;
    background: rgba(255, 255, 255, 0.08);
    box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2);
    backdrop-filter: blur(6px);
    transition: all 0.3s ease-in-out;
}

/* 🏆 Stylish Headings */
h1, h2, h3, label {
    color: #D1A3FF;  /* Soft Lavender */
    text-align: center;
    font-weight: 700;
    letter-spacing: 0.8px;
    text-transform: uppercase;
}

/* 💜 Violet Submit Button */
button {
    background: linear-gradient(135deg, #6A0DAD, #5A0B8B);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 8px;
    font-size: 16px;
    font-weight: bold;
    cursor: pointer;
    text-transform: uppercase;
    transition: all 0.3s ease-in-out;
}

button:hover {
    transform: scale(1.05);
    background: linear-gradient(135deg, #5A0B8B, #4A086C);
}

/* 🎭 Minimal Input Fields */
.gradio-container input[type="text"] {
    border: 2px solid #D1A3FF;
    padding: 10px;
    border-radius: 6px;
    font-size: 15px;
    background: rgba(255, 255, 255, 0.12);
    color: white;
    font-weight: bold;
    text-align: center;
    transition: border 0.3s ease-in-out, transform 0.2s ease-in-out;
}

.gradio-container input[type="text"]:focus {
    border-color: #6A0DAD;
    outline: none;
    transform: scale(1.03);
}

/* 📊 Chart & Plot Styling */
.gradio-container .gradio-plot {
    transition: transform 0.2s ease-in-out;
    border-radius: 10px;
    box-shadow: 0px 3px 8px rgba(0, 0, 0, 0.15);
}

.gradio-container .gradio-plot:hover {
    transform: scale(1.02);
}
/* 🎨 Artistic Background with Subtle Patterns */
html, body {
    min-height: 100vh;
    overflow-y: auto;
    overflow-x: hidden;
    background: url('https://www.toptal.com/designers/subtlepatterns/uploads/dark_embroidery.png'); /* Elegant Dark Pattern */
    background-size: cover;
    background-attachment: fixed;
    backdrop-filter: blur(4px);
}

/* 📦 Elegant Container with a Glassy Effect */
.gradio-container {
    width: 100%;
    min-height: 100vh;
    padding: 25px;
    border-radius: 12px;
    background: rgba(255, 255, 255, 0.08);
    box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2);
    backdrop-filter: blur(8px);
    transition: all 0.3s ease-in-out;
}

/* 🏆 Stylish Headings */
h1, h2, h3, label {
    color: #E0A6FF;  /* Soft Lavender */
    text-align: center;
    font-weight: 700;
    letter-spacing: 0.8px;
    text-transform: uppercase;
}

/* 💜 Violet-Themed Submit Button */
button {
    background: linear-gradient(135deg, #7D1FC3, #5D0F8F);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 8px;
    font-size: 16px;
    font-weight: bold;
    cursor: pointer;
    text-transform: uppercase;
    transition: all 0.3s ease-in-out;
}

button:hover {
    transform: scale(1.05);
    background: linear-gradient(135deg, #5D0F8F, #3F0864);
}

/* 🎭 Minimal Input Fields */
.gradio-container input[type="text"] {
    border: 2px solid #E0A6FF;
    padding: 10px;
    border-radius: 6px;
    font-size: 15px;
    background: rgba(255, 255, 255, 0.12);
    color: white;
    font-weight: bold;
    text-align: center;
    transition: border 0.3s ease-in-out, transform 0.2s ease-in-out;
}

.gradio-container input[type="text"]:focus {
    border-color: #7D1FC3;
    outline: none;
    transform: scale(1.03);
}

/* 📊 Chart & Plot Styling */
.gradio-container .gradio-plot {
    transition: transform 0.2s ease-in-out;
    border-radius: 10px;
    box-shadow: 0px 3px 8px rgba(0, 0, 0, 0.15);
}

.gradio-container .gradio-plot:hover {
    transform: scale(1.02);
}
"""  # Fixed: CSS should be a string

# Gradio UI
with gr.Blocks(css=custom_css) as iface:  # Using the defined custom_css
    gr.Markdown("# 📊 Student Performance Analysis")
    gr.Markdown("Enter a student's name to get performance insights with graphs and personalized recommendations.")

    student_input = gr.Textbox(label="Enter Student Name")
    submit_button = gr.Button("Submit")

    with gr.Row():
        cluster_id = gr.Textbox(label="Cluster ID")
        cluster_label = gr.Textbox(label="Cluster Label")

    recommendation_box = gr.Textbox(label="Recommendation")

    with gr.Row():  # ✅ Shows both graphs in a row
        performance_plot = gr.Plot(label="Arrears Count(S1-S5)")
        Sgpa = gr.Plot(label="Performance Trend (S1-S5)")

    submit_button.click(
        fn=get_student_info,
        inputs=student_input,
        outputs=[cluster_id, cluster_label, recommendation_box, performance_plot, Sgpa]
    )  # Fixed: Indentation

iface.launch(share=True)